In [26]:
%%bash
cat << 'EOF' > /content/config.sh
#!/bin/bash

export ROOTDIR="/content/exterieur"
export VIDEOSOURCE="gsplat/input/IMG_4797.MOV"
export IMAGESET="gsplat/input/perfume/video"
export INPUT_MODE="video"

#export NUM_FRAMES=150
export FPS=3

#branche dev
export GIT_BRANCH="dev"
export BASENAME="exterieur"

export PROFILE="gpu/fast"
#export PROFILE="gpu/balanced"
#export PROFILE="cpu/fast"
EOF

In [2]:
!cat /content/config.sh

#!/bin/bash

export ROOTDIR="/content/exterieur"
export VIDEOSOURCE="gsplat/input/IMG_4797.MOV"
export IMAGESET="gsplat/input/perfume/video"
export INPUT_MODE="video"

#export NUM_FRAMES=150
export FPS=3

#branche dev
export GIT_BRANCH="dev"
export BASENAME="exterieur"

export PROFILE="gpu/fast"
#export PROFILE="gpu/balanced"
#export PROFILE="cpu/fast"


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [27]:
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
!bash Miniforge3-Linux-x86_64.sh -b -p /usr/local/miniforge

# activer conda pour cette session notebook
import os
os.environ["PATH"] = "/usr/local/miniforge/bin:" + os.environ["PATH"]

!conda --version

ERROR: File or directory already exists: '/usr/local/miniforge'
If you want to update an existing installation, use the -u option.
conda 26.1.1


In [5]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
(wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh && \
chmod +x Miniconda3-latest-Linux-x86_64.sh  && \
bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda  && \
/usr/local/miniconda/bin/conda init bash)

skipping this stage


In [30]:
%%bash
set -e
source /content/config.sh

mkdir -p "$ROOTDIR"

if [ -n "$VIDEOSOURCE" ]; then
  SRC_VIDEO="/content/drive/MyDrive/$VIDEOSOURCE"

  if [ ! -f "$SRC_VIDEO" ]; then
    echo "❌ Video not found: $SRC_VIDEO"
  else
    echo "🎬 Copying video: $SRC_VIDEO"
    cp -f "$SRC_VIDEO" "$ROOTDIR/video.mp4"
  fi
fi

if [ -n "$IMAGESET" ]; then
  SRC_IMAGES="/content/drive/MyDrive/$IMAGESET"
  DST_IMAGES="$ROOTDIR/images"

  if [ ! -d "$SRC_IMAGES" ]; then
    echo "❌ Image dataset not found: $SRC_IMAGES"
  else
    echo "🖼️ Preparing image dataset: $SRC_IMAGES"

    mkdir -p "$DST_IMAGES"

    COUNT=$(find "$SRC_IMAGES" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | wc -l | tr -d ' ')
    if [ "$COUNT" -lt 2 ]; then
      echo "❌ Not enough images ($COUNT)"
      exit 1
    fi

    rm -f "$DST_IMAGES"/frame_*.png 2>/dev/null || true

    i=1
    for img in $(find "$SRC_IMAGES" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | sort); do
      printf -v idx "%05d" "$i"
      cp "$img" "$DST_IMAGES/frame_${idx}.png"
      i=$((i+1))
    done

    echo "✅ Dataset ready in $DST_IMAGES"
  fi
fi

🎬 Copying video: /content/drive/MyDrive/gsplat/input/IMG_4797.MOV
❌ Image dataset not found: /content/drive/MyDrive/gsplat/input/perfume/video


In [7]:
!git clone https://github.com/NicoIGN/video_to_ply.git
%cd video_to_ply

Cloning into 'video_to_ply'...
remote: Enumerating objects: 1158, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 1158 (delta 82), reused 128 (delta 49), pack-reused 993 (from 1)
Receiving objects: 100% (1158/1158), 724.41 KiB | 2.90 MiB/s, done.
Resolving deltas: 100% (732/732), done.
/content/video_to_ply


In [31]:
%%bash
cd /content/video_to_ply
source /content/config.sh
git stash save && git checkout $GIT_BRANCH && git pull

No local changes to save
Your branch is up to date with 'origin/dev'.
Already up to date.


Already on 'dev'


In [21]:
!RUN=1; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
( source /usr/local/miniforge/etc/profile.d/conda.sh && mamba env remove -y -n gsplat )

info     libmamba ****************** Backtrace Start ******************
debug    libmamba Loading configuration
trace    libmamba Compute configurable 'create_base'
trace    libmamba Compute configurable 'no_env'
trace    libmamba Compute configurable 'no_rc'
trace    libmamba Compute configurable 'rc_files'
trace    libmamba Compute configurable 'root_prefix'
trace    libmamba Get RC files configuration from locations up to HomeDir
trace    libmamba Configuration not found at '/root/.mambarc'
trace    libmamba Configuration not found at '/root/.mamba/mambarc.d'
trace    libmamba Configuration not found at '/root/.mamba/mambarc'
trace    libmamba Configuration not found at '/root/.mamba/.mambarc'
trace    libmamba Configuration not found at '/root/.config/mamba/mambarc.d'
trace    libmamba Configuration not found at '/root/.config/mamba/mambarc'
trace    libmamba Configuration not found at '/root/.config/mamba/.mambarc'
trace    libmamba Configuration not found at '/root/.condarc'
trac

In [22]:
!source /usr/local/miniforge/etc/profile.d/conda.sh && \
mamba env list | grep -q gsplat && \
cd /content/video_to_ply/ && \
mamba run -n gsplat mamba env update -n gsplat -f environment/conda_colab.yml --prune -y || \
mamba env create -n gsplat -f environment/conda_colab.yml -y

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
pytorch/linux-64                                            Using cache
pytorch/noarch                                              Using cache
nvidia/linux-64                                             Using cache
nvidia/noarch                                               Using cache
[+] 0.0s
[+] 0.0s


Transaction

  Prefix: /usr/local/miniforge/envs/gsplat

  Updating specs:

   - python=3.10
   - pip
   - rclone
   - tree
   - tqdm
   - cmake
   - ninja
   - pytorch=2.3.1
   - torchvision=0.18.1
   - torchaudio=2.3.1
   - pytorch-cuda=12.1
   - numpy<2
   - scipy
   - openimageio
   - ffmpeg
   - imageio
   - imageio-ffmpeg
   - colmap=3.8
   - faiss-cpu
   - plyfile
   - glew
   - qt-main


  Package                                      Version  Build                         Channel           Size
────────────────────────────────────────

In [23]:
%%bash
source /usr/local/miniforge/etc/profile.d/conda.sh

SITE_PACKAGES=$(mamba run -n gsplat python -c "import site; print(site.getsitepackages()[0])")
TARGET="$SITE_PACKAGES/SuperGluePretrainedNetwork"

if [ ! -d "$TARGET" ]; then
    git clone --depth 1 \
        https://github.com/magicleap/SuperGluePretrainedNetwork.git \
        "$TARGET"
else
    echo "SuperGluePretrainedNetwork already installed."
fi

Cloning into '/usr/local/miniforge/envs/gsplat/lib/python3.10/site-packages/SuperGluePretrainedNetwork'...


In [ ]:
!source /usr/local/miniforge/etc/profile.d/conda.sh && \
source /content/config.sh && unset NUM_FRAMES && \
echo INPUT_MODE=$INPUT_MODE && \
cd /content/video_to_ply/ && \
INPUT_ARG="" && \
if [ "$INPUT_MODE" = "images" ] && [ -n "$IMAGESET" ]; then \
  INPUT_ARG="--images $ROOTDIR/images --name $BASENAME"; \
elif [ "$INPUT_MODE" = "video" ] && [ -n "$VIDEOSOURCE" ]; then \
  INPUT_ARG="--video $ROOTDIR/video.mp4 --fps $FPS --name $BASENAME"; \
fi && \
mamba run -n gsplat bash run.sh $INPUT_ARG --root "$ROOTDIR" --skip-conda --profile "$PROFILE" --no-proxy

INPUT_MODE=video
🚫 Proxy disabled (NO_PROXY=true)
👉 using profile: gpu/fast
⏩ Skipping conda setup (--skip-conda enabled)
✅ Using python: Python 3.10.20 
🚀 GPU model OK: splatfacto
📦 ROOT: /content/exterieur
🎬 Extracting frames at 3 FPS → /content/exterieur/ori/images
🎬 Video:   /content/exterieur/input/video.mov
📏 Width:   1280px
📁 Output:  /content/exterieur/ori/images
⚙️ Mode:    Fixed FPS
🎞️ FPS:     3


In [13]:
%%bash
source /usr/local/miniforge/etc/profile.d/conda.sh && \
source /content/config.sh
mamba run -n gsplat  python -c "import torch; print(torch.version.cuda)"
mamba run -n gsplat  python -c "import torch; import torchvision; print(torch.__version__, torchvision.__version__)"

12.1
2.3.1 0.18.1


In [14]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
( source /usr/local/miniforge/etc/profile.d/conda.sh && \
  source /content/config.sh && \
  cd /content/video_to_ply/ && \
  INPUT_ARG="" && \
  if [ "$INPUT_MODE" = "images" ] && [ -n "$IMAGESET" ]; then \
    INPUT_ARG="--images $ROOTDIR/images --name $BASENAME"; \
  elif [ "$INPUT_MODE" = "video" ] && [ -n "$VIDEOSOURCE" ]; then \
    INPUT_ARG="--video $ROOTDIR/video.mp4 --name $BASENAME"; \
  fi && \
  mamba run -n gsplat bash run.sh $INPUT_ARG \
    --root "$ROOTDIR" \
    --skip-conda \
    --profile "$PROFILE" \
    --no-proxy \
    --skip-conda \
    --skip-frame-extraction \
    --skip-colmap \
    --skip-training )

skipping this stage


In [15]:
%%bash
set -e
source /content/config.sh
cd $ROOTDIR/ori
zip -r $ROOTDIR/exports/colmap_$BASENAME.zip ./colmap

bash: line 3: cd: /content/exterieur/ori: No such file or directory


CalledProcessError: Command 'b'set -e\nsource /content/config.sh\ncd $ROOTDIR/ori\nzip -r $ROOTDIR/exports/colmap_$BASENAME.zip ./colmap\n'' returned non-zero exit status 1.

In [ ]:
from google.colab import files
import os
import subprocess

# =========================
# LOAD CONFIG.SH VARIABLES
# =========================
result = subprocess.run(
    "source /content/config.sh && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

# =========================
# CONFIG
# =========================
rootdir = os.environ.get("ROOTDIR", "/content/work")
export_dir = os.path.join(rootdir, "exports")
basename = os.environ.get("BASENAME", "")

if not basename:
    print("❌ BASENAME is not set")
    raise SystemExit(1)

base_ply = os.path.join(export_dir, f"{basename}.ply")

# =========================
# EXPORT ORIGINAL PLY
# =========================
if os.path.exists(base_ply):
    print(f"⬇️ Downloading original PLY: {os.path.basename(base_ply)}")
    files.download(base_ply)
else:
    print(f"⚠️ Original PLY not found: {base_ply}")

In [ ]:
from google.colab import files
import os
import subprocess
import glob

# =========================
# LOAD CONFIG.SH VARIABLES
# =========================
result = subprocess.run(
    "source /content/config.sh && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

# =========================
# CONFIG
# =========================
rootdir = os.environ.get("ROOTDIR", "/content/work")
export_dir = os.path.join(rootdir, "exports")
basename = os.environ.get("BASENAME", "")

if not basename:
    print("❌ BASENAME is not set")
    raise SystemExit(1)

# =========================
# FIND FILTERED PLYS
# =========================
filtered_plys = sorted(
    glob.glob(os.path.join(export_dir, f"{basename}_*.ply"))
)

# =========================
# EXPORT FILTERED PLYS
# =========================
if not filtered_plys:
    print("⚠️ No filtered PLY files found.")
    print(f"📂 Searched: {export_dir}")
else:
    print(f"📦 Found {len(filtered_plys)} filtered PLY file(s)")

    for ply_path in filtered_plys:
        print(f"⬇️ Downloading filtered PLY: {os.path.basename(ply_path)}")
        files.download(ply_path)